# 붓꽃 분석 — 전처리 수정본
Iris1.csv와 Iris2.csv를 이 노트북과 같은 폴더에 두고, 커널을 재시작한 뒤 위에서부터 순서대로 실행하세요.
결측치 처리 및 종류별 IQR 이상치 후보 제거를 적용했습니다. IQR 후보가 반드시 측정 오류인 것은 아니므로 제거 목록을 확인하세요.
이 파일은 코드 및 구조 검증만 완료했으며, 원본 CSV가 없어 데이터 실행 결과는 검증하지 않았습니다. 기존 출력은 혼동을 막기 위해 지웠습니다.
주의: 여기서는 탐색·실습 목적으로 전체 데이터에서 IQR을 계산합니다. 엄밀한 예측 성능 평가에서는 먼저 훈련/테스트를 나누고 전처리 기준을 훈련 데이터에서만 구해야 합니다. 종류별 제거는 정답 종류를 활용하므로 순수한 비지도 군집 실험과도 구분하세요.


In [ ]:
import pandas as pd                              # 데이터 분석 및 조작
import numpy as np                               # 수치 연산
import matplotlib.pyplot as plt                  # 기본 그래프 그리기
import seaborn as sns

In [ ]:
df1 = pd.read_csv('Iris1.csv', encoding='cp949')
df2 = pd.read_csv('Iris2.csv', encoding='cp949')

In [ ]:
# 파일을 합칠 때 인덱스를 새로 부여합니다.
df = pd.concat([df1, df2], ignore_index=True)


In [ ]:
# 원-핫 인코딩 전에 실행하세요.
cols = ['꽃받침 길이', '꽃받침 너비', '꽃잎 길이', '꽃잎 너비']
required = cols + ['종류']

df[required] = df[required].replace(r'^\s*$', np.nan, regex=True)
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols] = df[cols].replace([np.inf, -np.inf], np.nan)
df['종류'] = df['종류'].str.strip()
df['종류'] = df['종류'].replace('', np.nan)

df_clean = df.dropna(subset=required).copy()
print('원본 행 수:', len(df))
print('결측치·변환 불가·무한대 제거 행 수:', len(df) - len(df_clean))

# 종류별 IQR: 네 측정값 중 하나라도 범위를 벗어나면 후보로 지정합니다.
grouped = df_clean.groupby('종류')[cols]
q1 = grouped.transform(lambda s: s.quantile(0.25))
q3 = grouped.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier_mask = ((df_clean[cols] < lower) | (df_clean[cols] > upper)).any(axis=1)

outliers = df_clean.loc[outlier_mask].copy()
print('IQR 이상치 후보 (제거 전 원본과 비교하세요):')
print(outliers.to_string())

# IQR 이상치는 오류가 확정된 값은 아닙니다. 이 노트북은 후보를 제거합니다.
df3 = df_clean.loc[~outlier_mask].copy().reset_index(drop=True)
print('IQR 기준 제거 행 수:', len(outliers))
print('최종 행 수:', len(df3))
print('남은 결측치:')
print(df3[required].isna().sum())


In [ ]:
!pip install koreanize-matplotlib

In [ ]:
import koreanize_matplotlib

In [ ]:
plt.figure(figsize=(20,4))
plt.subplot(1,4,1)
sns.histplot(data=df3, x='꽃받침 길이', hue='종류')

plt.subplot(1,4,2)
sns.histplot(data=df3, x='꽃받침 너비', hue='종류')

plt.subplot(1,4,3)
sns.histplot(data=df3, x='꽃잎 길이', hue='종류')

plt.subplot(1,4,4)
sns.histplot(data=df3, x='꽃잎 너비', hue='종류')

In [ ]:
sns.catplot(y='꽃받침 길이',x='종류',data=df3,hue='종류',kind='box')
sns.catplot(y='꽃받침 너비',x='종류',data=df3,hue='종류',kind='box')
sns.catplot(y='꽃잎 길이',x='종류',data=df3,hue='종류',kind='box')
sns.catplot(y='꽃잎 너비',x='종류',data=df3,hue='종류',kind='box')

In [ ]:
Q1=df3.groupby('종류')['꽃받침 길이'].quantile(0.25)

Q3=df3.groupby('종류')['꽃받침 길이'].quantile(0.75)

IQR=Q3-Q1
print(Q1)
print(Q3)
print(IQR)


In [ ]:
lower=Q1-1.5*IQR
uper=Q3+1.5*IQR
print(lower)
print(uper)

In [ ]:
df3[df3['종류']=='버시컬러']

In [ ]:
Q1=df3.groupby('종류')['꽃받침 너비'].quantile(0.25)

Q3=df3.groupby('종류')['꽃받침 너비'].quantile(0.75)

IQR=Q3-Q1
print(Q1)
print(Q3)
print(IQR)

In [ ]:
Q1=df3.groupby('종류')['꽃잎 길이'].quantile(0.25)

Q3=df3.groupby('종류')['꽃잎 길이'].quantile(0.75)

IQR=Q3-Q1
print(Q1)
print(Q3)
print(IQR)

In [ ]:
Q1=df3.groupby('종류')['꽃잎 너비'].quantile(0.25)

Q3=df3.groupby('종류')['꽃잎 너비'].quantile(0.75)

IQR=Q3-Q1
print(Q1)
print(Q3)
print(IQR)

In [ ]:
# 이상치 제거는 위 전처리 셀에서 df3에 반영했습니다.
# 중복 인덱스 기반의 수동 삭제는 사용하지 않습니다.
print("전처리 후 행 수:", len(df3))


In [ ]:
df3.info()


In [ ]:
X=df3[['꽃잎 길이','꽃잎 너비']]
y=df3['종류']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=7)
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model=KNeighborsClassifier()
model.fit(X_train,y_train)

In [ ]:
model.score(X_test,y_test)


In [ ]:
p=model.predict(X_test)
print(p)

In [ ]:
c_name=pd.unique(df3['종류'])
c_name

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report

In [ ]:
cm=confusion_matrix(y_test,p)


In [ ]:
df3 = pd.get_dummies(df3, columns=['종류'])

In [ ]:

x=df3[['꽃받침 너비','꽃잎 길이','꽃잎 너비','종류_버지니카','종류_버시컬러','종류_세토사']]


y=df3['꽃받침 길이']

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(x_train,y_train)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
y_pred = model.predict(x_test)
print('결정계수(R²):', r2_score(y_test, y_pred))
print('평균제곱오차(MSE):', mean_squared_error(y_test, y_pred))



군집 비지도 학습
```
# 코드로 형식 지정됨
```



In [ ]:
from sklearn.cluster import KMeans
M = KMeans(n_clusters=3, random_state=42, n_init=10)


In [ ]:
df8 = df3[['꽃잎 길이', '꽃잎 너비']].copy()


In [ ]:
M.fit(df8)

In [ ]:
df8['예측'] = M.predict(df8[['꽃잎 길이', '꽃잎 너비']])


In [ ]:
idx0=df8.loc[df8['예측']==0]
idx0

In [ ]:
c=M.cluster_centers_
c

In [ ]:

# 데이터: 예측한 군집별로 색 구분
sns.scatterplot(
    data=df8,
    x='꽃잎 길이',
    y='꽃잎 너비',
    hue='예측',
    palette='Set1'
)

# 군집 중심: 빨간 별표
plt.scatter(
    x=c[:, 0],
    y=c[:, 1],
    c='red',
    s=100,
    marker='*',
    label='군집 중심'
)

plt.legend()
plt.show()
